### Taller de IA_II Corte III
A continuacion van a encontrar un dataset sobre un censo que se realizó en estados unidos y hay diferentes tipos de variables para determinar si una persona va a ganar o no más de 50000 dólares anuales. 

In [44]:
import os
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo     # Importamos el dataset de la librería ucimlrepo

archivo = "census_income.csv"           # Creamos un dataset local para no tener que descargarlo cada vez que ejecutemos el código

if os.path.exists(archivo):             # Si el archivo existe, lo cargamos desde el disco
    df = pd.read_csv(archivo)               
else:                                   # Si el archivo no existe, lo descargamos y guardamos
    census_income = fetch_ucirepo(id=20)
    df = census_income.data.original
    df.to_csv(archivo, index=False)     # Guarda la copia

#X = df.drop(columns=["income"])         # Separamos el dataset en variables predictoras y variable objetivo
#y = df["income"]

In [ ]:
df.shape        # Mostramos la cantidad de registros y variables del dataset

(48842, 15)

In [ ]:
df.info()       # Mostramos información del dataset

<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             48842 non-null  int64
 1   workclass       47879 non-null  str  
 2   fnlwgt          48842 non-null  int64
 3   education       48842 non-null  str  
 4   education-num   48842 non-null  int64
 5   marital-status  48842 non-null  str  
 6   occupation      47876 non-null  str  
 7   relationship    48842 non-null  str  
 8   race            48842 non-null  str  
 9   sex             48842 non-null  str  
 10  capital-gain    48842 non-null  int64
 11  capital-loss    48842 non-null  int64
 12  hours-per-week  48842 non-null  int64
 13  native-country  48568 non-null  str  
 14  income          48842 non-null  str  
dtypes: int64(6), str(9)
memory usage: 5.6 MB


In [ ]:
df.isnull().sum()       # Contamos la cantidad de valores nulos por columna

age                 0
workclass         963
fnlwgt              0
education           0
education-num       0
marital-status      0
occupation        966
relationship        0
race                0
sex                 0
capital-gain        0
capital-loss        0
hours-per-week      0
native-country    274
income              0
dtype: int64

In [ ]:
(df == "?").sum()       # Contamos la cantidad de valores "?" por columna

age                  0
workclass         1836
fnlwgt               0
education            0
education-num        0
marital-status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital-gain         0
capital-loss         0
hours-per-week       0
native-country     583
income               0
dtype: int64

In [38]:
df["income"].unique()           # Mostramos las categorías de la variable objetivo 

<StringArray>
['<=50K', '>50K', '<=50K.', '>50K.']
Length: 4, dtype: str

In [39]:
df["income"].value_counts()     # Contamos la cantidad de registros por categoría de la variable objetivo

income
<=50K     24720
<=50K.    12435
>50K       7841
>50K.      3846
Name: count, dtype: int64

In [ ]:
""" Validamos la cantidad de registros por categoría de las variables categóricas
para identificar posibles errores de tipeo o inconsistencias en los nombres de las categorías.  """

columnas_texto = df.select_dtypes(exclude="number").columns

for col in columnas_texto:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


--- workclass ---
workclass
Private             33906
Self-emp-not-inc     3862
Local-gov            3136
State-gov            1981
?                    1836
Self-emp-inc         1695
Federal-gov          1432
NaN                   963
Without-pay            21
Never-worked           10
Name: count, dtype: int64

--- education ---
education
HS-grad         15784
Some-college    10878
Bachelors        8025
Masters          2657
Assoc-voc        2061
11th             1812
Assoc-acdm       1601
10th             1389
7th-8th           955
Prof-school       834
9th               756
12th              657
Doctorate         594
5th-6th           509
1st-4th           247
Preschool          83
Name: count, dtype: int64

--- marital-status ---
marital-status
Married-civ-spouse       22379
Never-married            16117
Divorced                  6633
Separated                 1530
Widowed                   1518
Married-spouse-absent      628
Married-AF-spouse           37
Name: count, dtype: in

In [43]:
# Validadmos si existen registros que solo contienen caracteres especiales en las variables de tipo texto

for col in columnas_texto:
    sospechosos = df[col][df[col].str.fullmatch(r"[^A-Za-z0-9]*", na=False)]
    if not sospechosos.empty:
        print(col, sospechosos.value_counts().to_dict())

workclass {'?': 1836}
occupation {'?': 1843}
native-country {'?': 583}


In [45]:
""" Teniendo en cuenta lo anterior limpiamos el dataset reemplazando los valores "?" por NaN 
y eliminando los registros que contienen caracteres especiales en las variables de tipo texto """

for col in columnas_texto:
    df[col] = df[col].str.strip()   # Eliminamos espacios en blanco al inicio y al final de los registros

df = df.replace("?", np.nan)        # Reemplazamos los valores "?" por NaN   

for col in ["workclass", "occupation", "native-country"]:
    df[col] = df[col].fillna("Desconocido")   # Reemplazamos los valores NaN por "Desconocido" en las variables categóricas

df["income"] = df["income"].str.replace(".", "", regex=False)   # Eliminamos el punto de los registros de la variable objetivo

print(df.isnull().sum())       # Contamos la cantidad de valores nulos por columna
print(df["income"].value_counts())     # Contamos la cantidad de registros por categoría de la variable objetivo

age               0
workclass         0
fnlwgt            0
education         0
education-num     0
marital-status    0
occupation        0
relationship      0
race              0
sex               0
capital-gain      0
capital-loss      0
hours-per-week    0
native-country    0
income            0
dtype: int64
income
<=50K    37155
>50K     11687
Name: count, dtype: int64


In [32]:
df.head(20)

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K
5,37,Private,284582,Masters,14,Married-civ-spouse,Exec-managerial,Wife,White,Female,0,0,40,United-States,<=50K
6,49,Private,160187,9th,5,Married-spouse-absent,Other-service,Not-in-family,Black,Female,0,0,16,Jamaica,<=50K
7,52,Self-emp-not-inc,209642,HS-grad,9,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,45,United-States,>50K
8,31,Private,45781,Masters,14,Never-married,Prof-specialty,Not-in-family,White,Female,14084,0,50,United-States,>50K
9,42,Private,159449,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,5178,0,40,United-States,>50K
